In [41]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

In [42]:
df = pd.read_csv("train.csv")

In [43]:
current_year = 2026

df["vehicle_age"] = current_year - df["model_year"]

df["mileage"] = (
    df["milage"]
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

df["mileage_per_year"] = (
    df["mileage"] / df["vehicle_age"].clip(lower=1)
)

df["horsepower"] = (
    df["engine"]
    .str.extract(r"(\d+(?:\.\d+)?)HP", expand=False)
    .astype(float)
)

df["engine_displacement"] = (
    df["engine"]
    .str.extract(r"(\d+(?:\.\d+)?)L", expand=False)
    .astype(float)
)

df["cylinders"] = (
    df["engine"]
    .str.extract(r"(\d+)\s*Cylinder", expand=False)
    .astype(float)
)

layout_cylinders = (
    df["engine"]
    .str.extract(r"\b[VHI](\d+)\b", expand=False)
    .astype(float)
)

df["cylinders"] = df["cylinders"].fillna(layout_cylinders)

df["hp_per_liter"] = (
    df["horsepower"] /
    df["engine_displacement"].replace(0, np.nan)
)

df["hp_per_cylinder"] = (
    df["horsepower"] /
    df["cylinders"].replace(0, np.nan)
)

df["is_electric"] = (
    df["engine"]
    .str.contains("Electric", case=False, na=False)
    .astype(int)
)

df["is_hybrid"] = (
    df["engine"]
    .str.contains("Hybrid", case=False, na=False)
    .astype(int)
)

df["is_turbo_engine"] = (
    df["engine"]
    .str.contains("Turbo", case=False, na=False)
    .astype(int)
)

In [44]:
df["transmission_gears"] = (
    df["transmission"]
    .str.extract(
        r"\b(\d+)[-\s]*(?:Speed|Spd)\b",
        expand=False
    )
    .astype(float)
)

df["is_automatic"] = (
    df["transmission"]
    .str.contains(
        r"A/T|Automatic",
        case=False,
        na=False
    )
    .astype(int)
)

df["is_manual"] = (
    df["transmission"]
    .str.contains(
        r"\bM/T\b|Manual",
        case=False,
        na=False
    )
    .astype(int)
)

df["is_cvt"] = (
    df["transmission"]
    .str.contains(
        "CVT",
        case=False,
        na=False
    )
    .astype(int)
)

In [45]:
df["has_accident"] = (
    ~df["accident"]
    .fillna("")
    .str.contains(
        "None reported",
        case=False
    )
).astype(int)

df["has_clean_title"] = (
    df["clean_title"]
    .fillna("No")
    .str.lower()
    .eq("yes")
    .astype(int)
)

In [46]:
df["is_turbo"] = (
    df["model"]
    .str.contains(
        r"\bTurbo\b",
        case=False,
        na=False
    )
    .astype(int)
)

df["is_sport"] = (
    df["model"]
    .str.contains(
        r"\bSport\b|\bGTS\b|\bGT\b|\bRS\b|\bSS\b|\bST\b",
        case=False,
        na=False
    )
    .astype(int)
)

df["is_limited"] = (
    df["model"]
    .str.contains(
        r"\bLimited\b|\bAnniversary\b|\bSpecial\b",
        case=False,
        na=False
    )
    .astype(int)
)

In [47]:
numeric_cols_poly = [
    "vehicle_age",
    "mileage",
    "mileage_per_year",
    "horsepower",
    "engine_displacement",
    "cylinders",
    "hp_per_liter",
    "hp_per_cylinder",
    "transmission_gears",
    "is_electric",
    "is_hybrid",
    "is_turbo_engine",
    "is_automatic",
    "is_manual",
    "is_cvt",
    "has_accident",
    "has_clean_title",
    "is_turbo",
    "is_sport",
    "is_limited"
]

In [48]:
X = df[numeric_cols_poly]
y = df["price"]

In [49]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [50]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=100))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

r2 = r2_score(y_val, y_pred)

print("Polynomial Ridge R²:", round(r2, 4))

Polynomial Ridge R²: 0.1877


In [51]:
alphas = [
    0.1,
    1,
    10,
    30,
    100,
    300,
    1000,
    3000,
    10000
]

model_1000 = None

for alpha in alphas:

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(
            degree=2,
            include_bias=False
        )),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    r2 = r2_score(y_val, y_pred)

    print(f"alpha={alpha:<6} R²={r2:.4f}")

    if alpha == 1000:
        model_1000 = model

alpha=0.1    R²=-0.0087
alpha=1      R²=0.0289
alpha=10     R²=0.0915
alpha=30     R²=0.1390
alpha=100    R²=0.1877
alpha=300    R²=0.2157
alpha=1000   R²=0.2231
alpha=3000   R²=0.2155
alpha=10000  R²=0.1989


In [52]:

test_df = pd.read_csv("test.csv")


current_year = 2026

test_df["vehicle_age"] = current_year - test_df["model_year"]

test_df["mileage"] = (
    test_df["milage"]
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

test_df["mileage_per_year"] = (
    test_df["mileage"] / test_df["vehicle_age"].clip(lower=1)
)

test_df["horsepower"] = (
    test_df["engine"]
    .str.extract(r"(\d+(?:\.\d+)?)HP", expand=False)
    .astype(float)
)

test_df["engine_displacement"] = (
    test_df["engine"]
    .str.extract(r"(\d+(?:\.\d+)?)L", expand=False)
    .astype(float)
)

test_df["cylinders"] = (
    test_df["engine"]
    .str.extract(r"(\d+)\s*Cylinder", expand=False)
    .astype(float)
)

layout_cylinders = (
    test_df["engine"]
    .str.extract(r"\b[VHI](\d+)\b", expand=False)
    .astype(float)
)

test_df["cylinders"] = test_df["cylinders"].fillna(layout_cylinders)

test_df["hp_per_liter"] = (
    test_df["horsepower"] /
    test_df["engine_displacement"].replace(0, np.nan)
)

test_df["hp_per_cylinder"] = (
    test_df["horsepower"] /
    test_df["cylinders"].replace(0, np.nan)
)

test_df["is_electric"] = (
    test_df["engine"].str.contains("Electric", case=False, na=False).astype(int)
)

test_df["is_hybrid"] = (
    test_df["engine"].str.contains("Hybrid", case=False, na=False).astype(int)
)

test_df["is_turbo_engine"] = (
    test_df["engine"].str.contains("Turbo", case=False, na=False).astype(int)
)

test_df["transmission_gears"] = (
    test_df["transmission"]
    .str.extract(r"\b(\d+)[-\s]*(?:Speed|Spd)\b", expand=False)
    .astype(float)
)

test_df["is_automatic"] = (
    test_df["transmission"]
    .str.contains(r"A/T|Automatic", case=False, na=False).astype(int)
)

test_df["is_manual"] = (
    test_df["transmission"]
    .str.contains(r"\bM/T\b|Manual", case=False, na=False).astype(int)
)

test_df["is_cvt"] = (
    test_df["transmission"]
    .str.contains("CVT", case=False, na=False).astype(int)
)

test_df["has_accident"] = (
    ~test_df["accident"].fillna("").str.contains(
        "None reported", case=False
    )
).astype(int)

test_df["has_clean_title"] = (
    test_df["clean_title"].fillna("No").str.lower().eq("yes").astype(int)
)

test_df["is_turbo"] = (
    test_df["model"].str.contains(r"\bTurbo\b", case=False, na=False).astype(int)
)

test_df["is_sport"] = (
    test_df["model"].str.contains(
        r"\bSport\b|\bGTS\b|\bGT\b|\bRS\b|\bSS\b|\bST\b",
        case=False,
        na=False
    ).astype(int)
)

test_df["is_limited"] = (
    test_df["model"].str.contains(
        r"\bLimited\b|\bAnniversary\b|\bSpecial\b",
        case=False,
        na=False
    ).astype(int)
)

In [53]:
numeric_cols = [
    "vehicle_age",
    "mileage",
    "mileage_per_year",
    "horsepower",
    "engine_displacement",
    "cylinders",
    "hp_per_liter",
    "hp_per_cylinder",
    "transmission_gears",
    "is_electric",
    "is_hybrid",
    "is_turbo_engine",
    "is_automatic",
    "is_manual",
    "is_cvt",
    "has_accident",
    "has_clean_title",
    "is_turbo",
    "is_sport",
    "is_limited"
]

categorical_cols = [
    "brand",
    "model",
    "fuel_type"
]

X_test = test_df[numeric_cols]

In [54]:
test_predictions = model_1000.predict(X_test)

In [55]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "price": test_predictions
})

submission.to_csv("submission.csv", index=False)